# Leereenheid 4.4: Data-integrasie

## Kombineer data vanaf verskeie bronne en los inkonsekwenthede en konflikte op

### Gevallestudie: Boland Meubels & Toestelle

Pieter het sy verkoopsdata, maar hy het ook winkel-inligting en kliënt-inligting in aparte lêers. Hy moet al hierdie bronne kombineer om volledige analise te doen.

In [ ]:
# Laai nodige biblioteke
import pandas as pd
import numpy as np
import urllib

## Stap 1: Laai al die datastelle

Ons het drie datastelle wat ons moet kombineer:
1. Verkope (van LU 4.3)
2. Winkels
3. Kliënte

In [ ]:
# Laai verkoopsdata (van LU 4.3)
url_verkope = 'https://raw.githubusercontent.com/aby-akademia/NGRDA150-2026/main/datastelle/le4_boland_meubels_verkope_verryk.csv'
verkope = pd.read_csv(url_verkope)
verkope['datum'] = pd.to_datetime(verkope['datum'])

print(f"Verkope: {len(verkope)} transaksies")
print(f"Kolomme: {verkope.columns.tolist()[:10]}...")  # Wys eerste 10 kolomme
verkope.head()

In [ ]:
# Laai winkel-inligting
url_winkels = 'https://raw.githubusercontent.com/aby-akademia/NGRDA150-2026/main/datastelle/le4_boland_meubels_winkels.csv'
winkels = pd.read_csv(url_winkels)

print(f"\nWinkels: {len(winkels)} rekords")
winkels

In [ ]:
# Laai kliënt-inligting
url_rou = 'https://raw.githubusercontent.com/aby-akademia/NGRDA150-2026/main/datastelle/le4_boland_meubels_kliënte.csv'

# Ons enkripteer slegs die deel met die spesiale karakters
# of gebruik quote vir die hele string (maar wees versigtig met die '://')
url_kliente = urllib.parse.quote(url_rou, safe=':/')

kliente = pd.read_csv(url_kliente)

print(f"\nKliënte: {len(kliente)} kliënte")
kliente.head()

## Stap 2: Identifiseer datakwaliteitsprobleme in winkels

Voor ons saamvoeg, moet ons kyk of daar probleme is.

In [ ]:
# Kyk vir duplikate in winkels datastel
print("Unieke winkelname in winkels datastel:")
print(winkels['winkel_naam'].value_counts())

# Kyk vir duplikate
duplikate = winkels.duplicated(subset=['winkel_naam'], keep=False)
if duplikate.sum() > 0:
    print(f"\nProbleem: {duplikate.sum()} duplikate winkel rekords gevind!")
    print(winkels[duplikate])

In [ ]:
# Kyk na winkelname in beide datastelle
print("Winkelname in verkope datastel:")
print(verkope['winkel_naam'].unique())

print("\nWinkelname in winkels datastel:")
print(winkels['winkel_naam'].unique())

## Stap 3: Skoonmaak van winkels datastel

Ons moet die probleme oplos voor ons kan saamvoeg.

In [ ]:
# Maak 'n skoon kopie
winkels_skoon = winkels.copy()

# Standaardiseer winkelname (verwyder spasies, maak hoofletters konsekwent)
winkels_skoon['winkel_naam'] = winkels_skoon['winkel_naam'].str.strip().str.title()

print("Na standaardisering:")
print(winkels_skoon['winkel_naam'].unique())

In [ ]:
# Verwyder duplikate (hou eerste voorkoms)
voor_verwyder = len(winkels_skoon)
winkels_skoon = winkels_skoon.drop_duplicates(subset=['winkel_naam'], keep='first')
na_verwyder = len(winkels_skoon)

print(f"Duplikate verwyder: {voor_verwyder - na_verwyder}")
print(f"Finale aantal winkels: {na_verwyder}")

# Wys finale skoon winkels datastel
winkels_skoon

## Stap 4: Voorbereiding vir samevoeging

Verseker dat die sleutel-kolomme ooreenstem.

In [ ]:
# Standaardiseer winkel_naam in verkope datastel ook
verkope['winkel_naam'] = verkope['winkel_naam'].str.strip().str.title()

# Kyk of al die winkels in verkope ook in winkels_skoon voorkom
verkope_winkels = set(verkope['winkel_naam'].unique())
winkels_lys = set(winkels_skoon['winkel_naam'].unique())

print("Winkels in verkope:", verkope_winkels)
print("Winkels in winkels datastel:", winkels_lys)

# Kyk vir winkels wat net in een datastel voorkom
net_in_verkope = verkope_winkels - winkels_lys
net_in_winkels = winkels_lys - verkope_winkels

if net_in_verkope:
    print(f"\nWaarskuwing: Winkels in verkope maar NIE in winkels datastel: {net_in_verkope}")
if net_in_winkels:
    print(f"Winkels in winkels datastel maar NIE in verkope: {net_in_winkels}")

## Stap 5: Samevoeging 1 - verkope + winkels

Ons voeg winkel-inligting by verkope.

In [ ]:
# Linker samevoeging: hou alle verkope, voeg winkel-inligting by waar beskikbaar
verkope_met_winkels = pd.merge(
    verkope,
    winkels_skoon,
    on='winkel_naam',
    how='left'
)

print(f"Voor samevoeging: {len(verkope)} transaksies")
print(f"Na samevoeging: {len(verkope_met_winkels)} transaksies")
print(f"\nNuwe kolomme van winkels:")
nuwe_kolomme = ['vloeroppervlak', 'aantal_personeel', 'openingsdatum', 'bestuurder', 'ligging']
for col in nuwe_kolomme:
    print(f"  - {col}")

In [ ]:
# Kyk of daar transaksies is sonder winkel-inligting
ontbrekend = verkope_met_winkels['vloeroppervlak'].isnull().sum()
if ontbrekend > 0:
    print(f"\nWaarskuwing: {ontbrekend} transaksies het nie winkel-inligting nie")
    print("Winkels sonder inligting:")
    print(verkope_met_winkels[verkope_met_winkels['vloeroppervlak'].isnull()]['winkel_naam'].unique())
else:
    print("\nGoed! Alle transaksies het winkel-inligting.")

In [ ]:
# Wys voorbeelde van die geïntegreerde data
verkope_met_winkels[['transaksie_id', 'winkel_naam', 'vloeroppervlak', 'aantal_personeel', 'bestuurder']].head()

## Stap 6: Skep nuwe kenmerke na samevoeging

Nou dat ons winkel-inligting het, kan ons nuwe kenmerke skep.

In [ ]:
# Bereken verkope per vierkante meter
verkope_met_winkels['verkope_per_m2'] = (
    verkope_met_winkels['totale_inkomste'] / verkope_met_winkels['vloeroppervlak']
)

# Wys beskrywende statistiek
print("Verkope per vierkante meter (per transaksie):")
print(verkope_met_winkels.groupby('winkel_naam')['verkope_per_m2'].describe())

In [ ]:
# Bereken totale verkope per winkel en dan verkope per werknemer
verkope_per_winkel = verkope_met_winkels.groupby('winkel_naam').agg({
    'totale_inkomste': 'sum',
    'aantal_personeel': 'first',  # dieselfde vir alle transaksies van daardie winkel
    'vloeroppervlak': 'first'
}).reset_index()

verkope_per_winkel['verkope_per_werknemer'] = (
    verkope_per_winkel['totale_inkomste'] / verkope_per_winkel['aantal_personeel']
)

print("\nVerkope per werknemer (totale verkope / aantal personeel):")
verkope_per_winkel[['winkel_naam', 'totale_inkomste', 'aantal_personeel', 'verkope_per_werknemer']]

## Stap 7: Samevoeging 2 - voeg kliënt-inligting by

Nou voeg ons kliënt-inligting by die geïntegreerde datastel.

In [ ]:
# Linker samevoeging: voeg kliënt-inligting by
verkope_volledig = pd.merge(
    verkope_met_winkels,
    kliente,
    on='klient_id',
    how='left'
)

print(f"Voor samevoeging: {len(verkope_met_winkels)} transaksies")
print(f"Na samevoeging: {len(verkope_volledig)} transaksies")
print(f"\nNuwe kolomme van kliënte:")
klient_kolomme = ['naam', 'ouderdom', 'pos_kode', 'registrasie_datum']
for col in klient_kolomme:
    print(f"  - {col}")

In [ ]:
# Kyk of daar transaksies is sonder kliënt-inligting
ontbrekend_kliente = verkope_volledig['naam'].isnull().sum()
if ontbrekend_kliente > 0:
    print(f"\nWaarskuwing: {ontbrekend_kliente} transaksies het nie kliënt-inligting nie")
else:
    print("\nGoed! Alle transaksies het kliënt-inligting.")

In [ ]:
# Wys voorbeelde van die finale geïntegreerde data
verkope_volledig[['transaksie_id', 'winkel_naam', 'klient_id', 'naam', 'ouderdom', 'vloeroppervlak', 'aantal_personeel']].head()

## Stap 8: Finale geïntegreerde datastel

Kom ons kyk na die finale datastel met al die geïntegreerde inligting.

In [ ]:
print("FINALE GEÏNTEGREERDE DATASTEL")
print("=" * 50)
print(f"Aantal transaksies: {len(verkope_volledig)}")
print(f"Aantal kolomme: {len(verkope_volledig.columns)}")
print(f"\nDatabronne geïntegreer:")
print("  1. Verkope (oorspronklik)")
print("  2. Winkels (vloeroppervlak, personeel, bestuurder)")
print("  3. Kliënte (naam, ouderdom, pos_kode)")
print(f"\nOntbrekende waardes: {verkope_volledig.isnull().sum().sum()}")

In [ ]:
# Wys die finale datastel
verkope_volledig.head()

## Stap 9: Analiseer die geïntegreerde data

Nou kan ons vrae beantwoord wat nie moontlik was voor integrasie nie.

In [ ]:
# Vraag 1: Watter winkel het die beste verkope per vierkante meter?
prestasie_per_winkel = verkope_volledig.groupby('winkel_naam').agg({
    'totale_inkomste': 'sum',
    'vloeroppervlak': 'first',
    'aantal_personeel': 'first'
}).reset_index()

prestasie_per_winkel['totale_verkope_per_m2'] = (
    prestasie_per_winkel['totale_inkomste'] / prestasie_per_winkel['vloeroppervlak']
)

print("Winkel prestasie (verkope per vierkante meter):")
print(prestasie_per_winkel.sort_values('totale_verkope_per_m2', ascending=False))

In [ ]:
# Vraag 2: Watter ouderdomsgroep koop die meeste?
# Skep ouderdomsgroepe
verkope_volledig['ouderdomsgroep'] = pd.cut(
    verkope_volledig['ouderdom'],
    bins=[0, 30, 40, 50, 100],
    labels=['<30', '30-39', '40-49', '50+']
)

verkope_per_ouderdom = verkope_volledig.groupby('ouderdomsgroep', observed=False).agg({
    'totale_inkomste': ['sum', 'mean', 'count']
}).round(2)

print("\nVerkope per ouderdomsgroep:")
print(verkope_per_ouderdom)

In [ ]:
# Vraag 3: Watter geografiese areas (pos_kode) genereer die meeste inkomste?
verkope_per_area = verkope_volledig.groupby('pos_kode')['totale_inkomste'].agg(['sum', 'count']).round(2)
verkope_per_area = verkope_per_area.sort_values('sum', ascending=False)

print("\nVerkope per area (pos_kode):")
print(verkope_per_area.head(10))

## Stap 10: Stoor die finale geïntegreerde Data

Stoor die volledige geïntegreerde datastel vir verdere analise.

In [ ]:
# Stoor die finale geïntegreerde datastel
verkope_volledig.to_csv('boland_meubels_volledig_geintegreer.csv', index=False)
print("Finale geïntegreerde data gestoor as: boland_meubels_volledig_geintegreer.csv")

## Opsomming

In hierdie notaboek het ons:
1. Drie aparte datastelle gelaai (verkope, winkels, kliënte)
2. Datakwaliteitsprobleme geïdentifiseer in die winkels datastel (duplikate, inkonsistente name)
3. Die winkels datastel skoongemaak (standaardiseer name, verwyder duplikate)
4. Sleutel-kolomme voorberei vir samevoeging
5. Linker samevoeging gebruik om winkel-inligting by verkope te voeg
6. Nuwe kenmerke geskep na samevoeging (verkope per m2, verkope per werknemer)
7. Linker samevoeging gebruik om kliënt-inligting by te voeg
8. Die finale geïntegreerde datastel geanaliseer om vrae te beantwoord wat nie moontlik was voor integrasie nie

Pieter kan nou:
- Winkel-prestasie vergelyk (verkope per m2, verkope per werknemer)
- Verstaan watter ouderdomsgroepe die meeste koop
- Geografiese patrone identifiseer (pos_kode)
- Volledige analise doen met al sy data in een plek

Die geïntegreerde datastel is gereed vir gevorderde analise en verslae.